# SimulacraBench 教程

我们将使用示例模式
`data/sample.json` 来理解任务的形态，以及举办本竞赛更技术性的理由。提交约定、评分规则和竞赛规则见 `README.md`。

In [1]:
import os
import sys
from pathlib import Path

# 本笔记本位于 tutorials/ 目录，但工具代码位于代码库根目录，
# 下面所有路径（config.yml、data/sample.json、_sandbox/）都是相对该根目录写的。
# 先定位到根目录再工作，这样无论 Jupyter 是在本目录还是上一级目录启动，
# 笔记本的行为都一致。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "make_sandbox.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在代码库的克隆目录中运行本笔记本")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

# 与 score.py 使用的是同一批模块。这里没有任何东西重新实现
# 评分程序：当你在本笔记本中评分时，调用的正是给你评分的那段代码。
from make_sandbox import (ROLE_COLUMN, generated_items, load_config,
                          load_schema, options_for, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)

---

## 1. 任务的形态

`make_sandbox.py` 把一份模式变成与真实数据形态完全一致的数据集：相同的列、
相同的选项、相同的跳转逻辑。边际分布和变量间的依赖关系都是虚构的，因此在这里
调通的流程可以迁移，而在这里调参的模型不能。

它写出 `respondents.parquet`——每一位受访者，外加说明其用途的 `role` 列——
以及 `schema.json`，也就是你的 `predict()` 收到的东西。角色只在这里确定一次
并写入磁盘。`score.py` 只是读取它们，自己从不做任何划分。

In [2]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

respondents = load_frames("_sandbox/sample", sample)
print(sample["dataset"]["description"])
print()

MEANING = {"TRAIN": "两个阶段都可见",
           "DEV": "第 1 阶段评分，第 2 阶段可见",
           "FINAL": "第 2 阶段评分"}
counts = respondents[ROLE_COLUMN].value_counts()
print(pd.DataFrame({"受访者数": counts,
                    "含义": [MEANING[r] for r in counts.index]}).to_string())

A toy instrument, not a real survey. Ten items, few enough to print the whole schema and read it. It has one of everything the real schemas have: a frame block that is always visible, items that are scored, a gate chain two deep, and an EXCLUDE column the grader never shows anybody. The GIVEN block is deliberately the cheap half of a questionnaire -- the variables that already sit on a sampling frame, a census roster or another survey of the same households -- and the PREDICT block is the expensive half, the part that needs an enumerator and an interview. Use it to see the shape of the task; use the three real schemas to see whether a method works.

       受访者数                 含义
role                          
TRAIN  8981            两个阶段都可见
FINAL  2112           第 2 阶段评分
DEV     907  第 1 阶段评分，第 2 阶段可见


### 模式声明了什么

每个题目有四个键：原样的 `question`、一个 `class`、允许的 `values`，以及仅在
部分人被问到时才有的 `gate`。

这三个类别就是任务的全部。**`GIVEN`** 对所有人可见，从不评分。**`PREDICT`**
对被隐藏的受访者隐藏，且每一个空格都要评分。**`EXCLUDE`**——标识符、记录键、
自由文本——根本不会出现在数据表中，所以请按 `class` 过滤，不要假定模式和数据表
带有相同的列。

在这份问卷里，划分刻意把问卷中便宜的一半与昂贵的一半对立起来。`GIVEN` 部分是
抽样框、人口普查名册或针对同一批住户的另一项调查中已有的那类变量：住在哪里、
家庭规模多大、有没有手机。`PREDICT` 部分则需要调查员上门访谈。第 2 部分正是
建立在这一区别之上。

In [3]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"题目": name,
                 "类别": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "跳转依据": gate.get("parent", "-"),
                 "何时提问": ", ".join(gate.get("observed_if", [])) or "-",
                 "选项": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))

                    题目      类别  K           跳转依据                                                  何时提问                                                      选项
                region   GIVEN  3              -                                                     -                                 North | Central | South
           urban_rural   GIVEN  2              -                                                     -                                           Urban | Rural
              age_band   GIVEN  4              -                                                     -                             18-29 | 30-44 | 45-59 | 60+
        household_size   GIVEN  4              -                                                     -                               1 | 2-3 | 4-5 | 6 or more
household_has_children   GIVEN  2              -                                                     -                                                Yes | No
      has_mobile_phone   GIVEN  2             

`K` 是题目的选项个数，**并计入跳转哨兵值**：有跳转的题目比它的答案多出一格，
因为“从未被问到”对它而言是一个真正的答案。这一格是你概率向量的最后一项，
均匀参考值 `U` 也正是由 `K` 计算得出的。

`would_return` 依赖于 `clinic_wait`，后者又依赖于 `visited_clinic`：一条深度
为二的链。从未去过诊所的受访者，既没有被问过等了多久，也没有被问过是否会再去。
他们对这两题的真实答案都是 `NA_GATED`。

In [4]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(respondents[chain].value_counts().to_frame("受访者数").head(12).to_string())

                                                         受访者数
visited_clinic       clinic_wait           would_return      
No                   NA_GATED              NA_GATED      5179
Prefer not to answer NA_GATED              NA_GATED      4413
Yes                  Over 2 hours          Yes           1405
                     Under 30 minutes      Yes            380
                     Over 2 hours          No             204
                                           Not sure       201
                     Under 30 minutes      Not sure       162
                                           No              28
                     30 minutes to 2 hours Yes             21
                                           Not sure         4
                                           No               3


可以把这张表直接当作跳转逻辑本身来读：只要 `visited_clinic` 不是 `Yes`，
两个子题目就都是 `NA_GATED`，无一例外。**只要父题目可见，带跳转题目的答案就是
确定的**——这是白送的分数，也是首先应当利用的地方。

### `predict()` 收到什么

`score.py` 取出本阶段的可见角色与隐藏角色，把可见受访者叠放在隐藏受访者之上，
并清空后者的每一个 `PREDICT` 单元格。你要为这些空格返回概率。

`NaN` 只有一个含义：*这个单元格被隐藏了，请预测它*。它绝不表示“此人没有回答”
——真正的拒答是一个普通选项，例如 `Prefer not to answer`，和其他选项一样列在
选项表中。

In [5]:
frame, cells, truth = sample_rows(sample, respondents, config, PHASE, seed=SEED)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]

print("数据表:", frame.shape, " 待预测单元格:", len(cells))
print()
print(pd.concat([frame[["respondent_id"] + shown].head(3),
                 frame[["respondent_id"] + shown].tail(3)]).to_string(index=False))

数据表: (3564, 11)  待预测单元格: 3628

respondent_id  region urban_rural age_band household_size household_has_children has_mobile_phone visited_clinic clinic_wait would_return trusts_health_advice
      R000002   North       Rural      60+            4-5                    Yes               No             No    NA_GATED     NA_GATED           Not at all
      R000003   South       Rural      60+            2-3                     No               No             No    NA_GATED     NA_GATED                A lot
      R000005   South       Rural    18-29              1                    Yes               No             No    NA_GATED     NA_GATED                A lot
      R011985 Central       Urban    45-59            4-5                     No               No            NaN         NaN          NaN                  NaN
      R011987   South       Rural      60+            2-3                    Yes               No            NaN         NaN          NaN                  NaN
      R011994 C

上面几行是可见受访者：内容完整，供你学习。下面几行被隐藏——你只能看到他们的
`GIVEN` 部分，别无其他。

你的返回值是每个空格一个概率向量，按**规范顺序**排列：行从上到下，行内则按
`schema["items"]` 的键顺序排列题目——而不是 `frame.columns` 的顺序，两者可能
不同。每个向量依次对应该题目的 `values`，若该题目有跳转，再加上哨兵那一格。
顺序要从模式中读取，绝不要从数据中推断：没有人选过的选项同样占一格。

In [6]:
print(pd.DataFrame(cells, columns=["行", "respondent_id", "题目"]).head(8)
      .to_string(index=False))

   行 respondent_id                   题目
2657       R000011       visited_clinic
2657       R000011          clinic_wait
2657       R000011         would_return
2657       R000011 trusts_health_advice
2658       R000022       visited_clinic
2658       R000022          clinic_wait
2658       R000022         would_return
2658       R000022 trusts_health_advice


### 评分

群体基线：每位被隐藏的受访者都得到该题目经过平滑的整体比例，完全不考虑个体
差异。`skill` 在均匀猜测时为 0，完美时为 1，排行榜正是按它排序的。

In [7]:
def hidden_cells(frame, items):
    '''所有空格，按 predict() 必须返回的顺序排列。'''
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]


def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]


vectors = floored(crowd_for(sample, frame), sample, cells, config["scoring"]["floor"])
result = grade(sample, config, vectors, truth, cells)

def table(rows):
    """Print label/value pairs, aligned however long the labels happen to be."""
    pad = max(len(label) for label, _ in rows)
    for label, value in rows:
        print("%-*s  %s" % (pad, label, value))


table([("均匀参考值（奈特）", "%.4f" % result["uniform_reference"]),
       ("对数得分", "%.4f" % result["log_score"]),
       ("skill", "%.4f" % result["skill"])])
print()
print("skill 在均匀猜测时为 0，完美时为 1。")

均匀参考值（奈特）  1.3144
对数得分       -0.8941
skill      0.3198

skill 在均匀猜测时为 0，完美时为 1。


这就是全部约定。一份提交就是一个含有 `predict()` 的 `main.py`，它返回上述
向量；`score.py` 会按评分程序的方式运行它，而 `tools/check_submission_zip.py`
则检查你上传的压缩包格式是否正确。

---

## 2. 好模型能带来什么

一个奖励“预测他人回答”的基准，自然会引出一个疑问：目的是不是不再去问人了？
这里我们会看到一种把算法预测与人工样本结合起来的做法。

我们想得到关于某个总体的一个数字：信任本地诊所健康建议的住户比例。
便宜的那部分变量——地区、城乡、家庭规模、有无手机——凭借行政记录或早先的调查，
对抽样框中的每一户都已知。昂贵的那部分需要调查员上门，而预算只够几百次访谈。

你有三个选择。

1. **只做访谈。** 访问 300 户，取比例，报告置信区间。结果有效，精度也就是
   300 次访谈所能达到的水平。
2. **只用模型。** 用模型跑一遍每户的便宜变量，报告平均值。免费，而且*模型错
   多少就错多少*——没有区间，也无从知晓。
3. **两者并用。** 到处使用模型，再用这 300 次访谈来测量并扣除模型的误差。
   这就是**预测驱动推断**，也是本节余下部分要构建的东西。

值得采用的是第三种，因为无论模型好坏它都有效，而且*在模型好的时候比第一种更
精确*。

In [8]:
# 被估计量：至少在一定程度上信任健康建议的比例。
TARGET, POSITIVE = "trusts_health_advice", ["Somewhat", "A lot"]
GIVEN = [n for n, r in sample["items"].items() if r["class"] == "GIVEN"]

Y = respondents[TARGET].isin(POSITIVE).to_numpy(float)
design = pd.get_dummies(respondents[GIVEN].astype(str), drop_first=True)
X = np.column_stack([np.ones(len(design)), design.to_numpy(float)])

# 模型是在往期受访者上拟合的——即 TRAIN 角色，
# 也正是一份提交所能学习的可见部分。它从未见过我们即将访谈的那些住户。
past = (respondents[ROLE_COLUMN] == "TRAIN").to_numpy()
ridge = np.linalg.solve(X[past].T @ X[past] + 5 * np.eye(X.shape[1]),
                        X[past].T @ Y[past])
predicted = X @ ridge          # f(便宜变量)，对每一户

# 我们要给出数字的那部分：未用于拟合模型的住户。
frame_rows = np.flatnonzero(~past)
TRUTH = Y[frame_rows].mean()   # 只因数据是虚构的才可知

table([("模型拟合所用的往期受访者数:", "%d" % past.sum()),
       ("待估计的住户数:", "%d" % len(frame_rows)),
       ("预测值与真实回答的相关系数:", "%.2f"
        % np.corrcoef(predicted[frame_rows], Y[frame_rows])[0, 1]),
       ("真实比例（真实调查永远看不到）:", "%.3f" % TRUTH)])

模型拟合所用的往期受访者数:    8981
待估计的住户数:          3019
预测值与真实回答的相关系数:    0.57
真实比例（真实调查永远看不到）:  0.597


现在抽取这 300 次访谈，把三个数字都算出来。

“只做访谈”的区间就是教科书上的那个。预测驱动区间则是模型在你**没有**访谈的
住户上的平均值，再用模型在你已访谈住户上的平均误差加以校正：

```
估计 = 均值(预测 | 未访谈) - [ 均值(预测 | 已访谈) - 均值(回答 | 已访谈) ]
              ↑ 到处使用的模型            ↑ 实测出的模型误差
```

方括号里的这一项就是全部的安全机制。它由真实回答算出，因此要花掉真实的访谈，
并且无论模型的偏误是什么，它都会把偏误消掉。

In [9]:
def estimates(f, interviewed, rest):
    '''只做访谈与预测驱动两种估计，各自带标准误。'''
    y = Y[interviewed]
    classical = (y.mean(), y.std(ddof=1) / np.sqrt(len(y)))

    correction = f[interviewed].mean() - y.mean()
    powered = (f[rest].mean() - correction,
               np.sqrt(f[rest].var(ddof=1) / len(rest)
                       + (f[interviewed] - y).var(ddof=1) / len(interviewed)))
    return classical, powered


def band(estimate):
    point, se = estimate
    return "%.3f  [%.3f, %.3f]  宽度 %.3f" % (
        point, point - 1.96 * se, point + 1.96 * se, 2 * 1.96 * se)


N_INTERVIEWS = 300
draw = np.random.default_rng(SEED).permutation(frame_rows)
interviewed, rest = draw[:N_INTERVIEWS], draw[N_INTERVIEWS:]

classical, powered = estimates(predicted, interviewed, rest)
table([("真值", "%.3f" % TRUTH),
       ("只做访谈", band(classical)),
       ("预测驱动", band(powered)),
       ("只用模型（不做访谈）", "%.3f  [根本没有区间]"
        % predicted[frame_rows].mean())])

真值          0.597
只做访谈        0.620  [0.565, 0.675]  宽度 0.110
预测驱动        0.609  [0.560, 0.658]  宽度 0.099
只用模型（不做访谈）  0.596  [根本没有区间]


抽一次什么也证明不了——那个区间可能只是运气好。真正重要的是在许多次调查中的
表现：区间是否约有 95% 的时候包含真值，它有多宽？把整个过程重复一千次，每次
都换一批新的 300 户。

In [10]:
def repeat(f, n_interviews=N_INTERVIEWS, draws=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(draws):
        shuffled = rng.permutation(frame_rows)
        classical, powered = estimates(f, shuffled[:n_interviews],
                                       shuffled[n_interviews:])
        out.append(classical + powered)
    return np.array(out)          # 估计值、标准误、估计值、标准误


def summarize(trials, label):
    rows = []
    for name, point, se in (("只做访谈", trials[:, 0], trials[:, 1]),
                            ("预测驱动", trials[:, 2], trials[:, 3])):
        rows.append({"方法": name,
                     "平均宽度": (2 * 1.96 * se).mean(),
                     "包含真值的比例": np.mean(np.abs(point - TRUTH) <= 1.96 * se)})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False, float_format="%.3f"))
    return out


good = summarize(repeat(predicted), "一个预测得不错的模型")
narrower = 1 - good.loc[1, "平均宽度"] / good.loc[0, "平均宽度"]
print("\n区间窄了 %.0f%%，而访谈次数仍是同样的 %d 次。" % (100 * narrower, N_INTERVIEWS))
print("若只靠访谈买到这样的精度，大约需要 %d 次。" % round(N_INTERVIEWS / (1 - narrower) ** 2))

一个预测得不错的模型
  方法  平均宽度  包含真值的比例
只做访谈 0.111    0.961
预测驱动 0.093    0.964

区间窄了 16%，而访谈次数仍是同样的 300 次。
若只靠访谈买到这样的精度，大约需要 424 次。


两个区间都约有 95% 的时候包含真值——这正是它们之所以是区间的原因。预测驱动的
那个只是**更窄**，而实地工作量完全相同。最后一行才是整个练习的要点：更好的
模型不会从预算里减掉访谈，而是让每一次访谈更值钱。

### 模型很糟糕时会怎样

显而易见的反驳是：这只在模型正确时才成立，而信任模型正是风险所在。下面是同样的流程，但模型拟合在**另一个总体**上。

In [11]:
from make_sandbox import make_sandbox

elsewhere = make_sandbox(sample, config, seed=99)      # 另一个总体
other_design = pd.get_dummies(elsewhere[GIVEN].astype(str), drop_first=True)
other_X = np.column_stack([np.ones(len(other_design)), other_design.to_numpy(float)])
other_Y = elsewhere[TARGET].isin(POSITIVE).to_numpy(float)

wrong = np.linalg.solve(other_X.T @ other_X + 5 * np.eye(other_X.shape[1]),
                        other_X.T @ other_Y)
mispredicted = X @ wrong

table([("预测值与真实回答的相关系数:", "%.2f"
        % np.corrcoef(mispredicted[frame_rows], Y[frame_rows])[0, 1]),
       ("只用模型（不做访谈）", "%.3f   对比真值 %.3f   <- 偏差 %+.3f"
        % (mispredicted[frame_rows].mean(), TRUTH,
           mispredicted[frame_rows].mean() - TRUTH))])
print()
summarize(repeat(mispredicted), "一个无法迁移的模型")

预测值与真实回答的相关系数:  0.09
只用模型（不做访谈）      0.725   对比真值 0.597   <- 偏差 +0.129



一个无法迁移的模型
  方法  平均宽度  包含真值的比例
只做访谈 0.111    0.961
预测驱动 0.116    0.960


,方法,平均宽度,包含真值的比例
0,只做访谈,0.111063,0.961
1,预测驱动,0.116003,0.960


请注意：只用模型的估计偏差超过十分之一，而输出中没有任何东西会提醒你——没有
区间，没有警告，只有一个看起来和正确答案同样权威的数字。用模型取代实地工作会
引入偏误。

预测驱动区间仍然约有 95% 的时候包含真值。它并不比单纯访谈更窄——无用的模型买
不到精度。校正项在 300 次真实访谈上测出了模型的误差并将其扣除，这正是它存在
的意义。

### 为什么这需要一个基准

区间的宽度是模型好坏的直接函数。这正是要在真实问卷上、用严格的评分规则认真
衡量预测质量的理由：排行榜上更高的 `skill`，就是实地更窄的置信区间，或者用
更少的访谈得到同样的区间。